# Mode A LoRA Finetune — the *fair* text-side counterpart to Mode C

**What this notebook does:** trains a LoRA on `unsloth/Llama-3.2-1B-Instruct` for the
exact task the eval measures — maximise the probability of the correct letter token
(` A`..` J`) at the `Answer:` position of a Mode A ranking prompt — then evaluates it on
the same 604 held-out test users as Mode C.

**Why it is the fair comparison to Mode C's `adapter_ranking.pt`:**

| | Mode A LoRA (this nb) | Mode C ranking adapter |
|---|---|---|
| Training data | `build_train_ranking_examples` (same seed) | same |
| Prompt format | `render_mode_a_prompt` (same as eval) | same family |
| Loss | CE over ` A`..` J` letter logits at `Answer:` | same |
| Eval | `eval_ranking.py --modes A` on `test_ranking_prompts.json` | `--modes C` on same |
| Best ckpt | val HR@1 | val HR@1 |

The only remaining variables are candidate **modality** (text titles vs soft tokens) and
**where** learning lives (LoRA on the LLM vs the projector). That is the question we set out
to answer, so this is now an apples-to-apples comparison rather than trained-vs-zero-shot.

## 1. Clone repo + install deps

The repo carries every artifact the eval needs (`train.csv`, `test_ranking*.{csv,json}`,
`checkpoints/id_maps.json`, `checkpoints/item_embeddings.pt`, and the trained
`adapter_ranking.pt` for the Mode C side-by-side), so no data regeneration is required.

In [ ]:
# Clone the project (switch BRANCH if needed)
BRANCH = 'Adi'
!git clone --branch $BRANCH https://github.com/Krigupt/ECS172.git
%cd ECS172

# Deps: repo requirements + PEFT/TRL stack
!pip install -q -r requirements.txt
!pip install -q peft accelerate bitsandbytes

# Llama-3.2 is gated — log in (unsloth mirror usually does not require it, but be safe)
from huggingface_hub import login
login()

## 2. Train the Mode A ranking LoRA

Defaults mirror the ranking adapter (same seed → same candidate sets). `--max-train 5000`
gives the LoRA a comparable example budget; bump it if you want to spend more (the whole
point per our discussion: it's fine to give a method more budget as long as the *task* and
*eval* are identical). Chat template stays OFF to match `eval_ranking.py`'s default.

In [ ]:
!python scripts/finetune_lora_ranking.py \
    --model unsloth/Llama-3.2-1B-Instruct \
    --epochs 3 --max-train 5000 --val-fraction 0.1 \
    --lora-r 16 --lora-alpha 32 --lr 1e-4 --batch-size 8 \
    --output ./llama31-1b-movielens-ranking-lora

## 3. Evaluate Mode A (LoRA) on the held-out test users

Same scorer, same `test_ranking_prompts.json`, same metric as every other condition.

In [ ]:
# Mode A with the finetuned LoRA
!python scripts/eval_ranking.py --modes A \
    --model ./llama31-1b-movielens-ranking-lora \
    --n-candidates 10

## 4. (Optional) Side-by-side: Mode A zero-shot, Mode A LoRA, Mode C ranking adapter

Runs all three against the same data so you can drop the numbers straight into `results.md`.

In [ ]:
# Mode A zero-shot (base instruct) + Mode C (ranking adapter) for reference
!python scripts/eval_ranking.py --modes A C \
    --model unsloth/Llama-3.2-1B-Instruct \
    --embedding-adapter checkpoints/adapter_ranking.pt \
    --projected-embeddings checkpoints/projected_embeddings_ranking.pt \
    --n-candidates 10

print('\nMode A above = zero-shot baseline.')
print('Compare with the LoRA Mode A from section 3 (trained text baseline)')
print('and Mode C (ranking adapter, soft tokens).')

## 5. Reading the result

- **Mode C ≥ Mode A (LoRA):** collaborative soft tokens beat text *even when text gets the
  same trained advantage* — the strongest version of the headline claim.
- **Mode A (LoRA) ≈ Mode C:** the gap in `results.md` was mostly the training advantage, not
  modality. Still a clean, honest finding.
- **Mode A (LoRA) ≫ Mode A (zero-shot):** confirms the LoRA actually learned the ranking
  task (sanity check that the comparison is meaningful).